# `nb_06` — Reduction Ladder for Code: Evaluation Suite & Model Comparison

> **Purpose**: Evaluation-only notebook. No training code. Evaluates all 4 model
> checkpoints identically on a fixed benchmark pool, then produces the master
> comparison table and 4 publication-quality figures.

## Models Under Test

| ID | Description | Checkpoint |
|:---|:------------|:-----------|
| **M1** | Raw Qwen2.5-Coder-1.5B-Instruct (no fine-tuning) | base model |
| **M2** | Vanilla SFT with QLoRA | `checkpoints/qlora_vanilla_adapter` |
| **M4** | Standard GRPO (arm01b) | `checkpoints/standard_grpo_final` |
| **M6** | **Inv-GRPO** — Invariance-Regularised GRPO (arm01) | `checkpoints/inv_grpo_final` |

## Benchmark Pool (Fixed, Immutable)

| Level | Dataset | Tasks | OOD? |
|:------|:--------|------:|:-----|
| L0 | HumanEval Standard | 164 | No |
| L1 | EvoEval Subtle | — | No |
| L2 | EvoEval Tool-Use | — | No |
| L3 | EvoEval Creative | — | No |
| L4 | EvoEval Difficult | — | No |
| L5 | EvoEval Combine | — | No |
| **Ctrl** | **LiveCodeBench Lite** | — | **Yes** |

*Ctrl level is temporally separated from training data (post-cutoff problems).*

---


## Cell 01 — Environment Initialisation

In [ ]:
import sys
import subprocess
import platform
import datetime

print(f"Python       : {sys.version}")
print(f"Platform     : {platform.platform()}")
print(f"Timestamp    : {datetime.datetime.utcnow().isoformat()}Z")

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_properties(0)
        print(f"GPU          : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB)")
        print(f"CUDA version : {torch.version.cuda}")
    else:
        print("GPU          : CPU-only mode")
    print(f"PyTorch      : {torch.__version__}")
except ImportError:
    print("PyTorch not installed — evaluation will run in dry-run mode")

# ── Add repo root to path ────────────────────────────────────────────────────
import os
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print(f"Repo root    : {REPO_ROOT}")

# ── Verify evaluation module ─────────────────────────────────────────────────
from src.evaluation import EvaluationSuite, BenchmarkRegistry, EvaluationReporter
print("\n✅ src.evaluation module imported successfully")


## Cell 02 — Benchmark Registry Load & Integrity Check

In [ ]:
from src.evaluation import BenchmarkRegistry

# Load registry (reads all JSONL files from data/ladder/)
registry = BenchmarkRegistry()

# Print integrity report
print(registry.integrity_report())
print()

# Task counts per level
counts = registry.task_counts()
total = sum(counts.values())
print(f"Total tasks loaded: {total}")
print(f"Available levels  : {registry.available_levels()}")

# Smoke-test: first task of L0
l0_tasks = registry.get_tasks("L0")
if l0_tasks:
    t = l0_tasks[0]
    print(f"\nSample L0 task:")
    print(f"  task_id : {t.task_id}")
    print(f"  level   : {t.ladder_level}")
    print(f"  prompt  : {t.prompt[:80].strip()}...")


## Cell 03 — M1: Raw Baseline (Qwen2.5-Coder-1.5B-Instruct, No Adapter)

This is the **zero-shot performance** of the unmodified pre-trained model.
All other models are compared against this baseline.


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
M1_OUTPUT  = "results/M1_baseline"

suite = EvaluationSuite(
    registry=registry,
    evaluate_pass5=True,
    timeout_seconds=10.0,
)

m1_report = suite.evaluate_model(
    model_id="M1_baseline",
    model_name_or_path=BASE_MODEL,
    adapter_path=None,
    output_dir=M1_OUTPUT,
)
print(f"\nM1 Ladder AUC     : {m1_report.ladder_auc*100:.2f}%")
print(f"M1 Collapse Point : {m1_report.collapse_point}")


## Cell 04 — M2: Vanilla SFT with QLoRA

Fine-tuned on correct HumanEval solutions only (no RL, no invariance objective).
Adapter: `checkpoints/qlora_vanilla_adapter`


In [ ]:
M2_ADAPTER = "checkpoints/qlora_vanilla_adapter"
M2_OUTPUT  = "results/M2_vanilla_sft"

m2_report = suite.evaluate_model(
    model_id="M2_vanilla_sft",
    model_name_or_path=BASE_MODEL,
    adapter_path=M2_ADAPTER,
    output_dir=M2_OUTPUT,
)
print(f"\nM2 Ladder AUC     : {m2_report.ladder_auc*100:.2f}%")
print(f"M2 Collapse Point : {m2_report.collapse_point}")


## Cell 05 — M4: Standard GRPO (arm01b)

Standard Group Relative Policy Optimisation with no invariance regularisation.
Adapter: `checkpoints/standard_grpo_final`

This is the **ablation control** that proves our invariance term adds value.


In [ ]:
M4_ADAPTER = "checkpoints/standard_grpo_final"
M4_OUTPUT  = "results/M4_standard_grpo"

m4_report = suite.evaluate_model(
    model_id="M4_standard_grpo",
    model_name_or_path=BASE_MODEL,
    adapter_path=M4_ADAPTER,
    output_dir=M4_OUTPUT,
)
print(f"\nM4 Ladder AUC     : {m4_report.ladder_auc*100:.2f}%")
print(f"M4 Collapse Point : {m4_report.collapse_point}")


## Cell 06 — M6: Inv-GRPO (arm01 — Our Method)

**Invariance-Regularised GRPO** with paired perturbation sampling.
Adapter: `checkpoints/inv_grpo_final`

This is the proposed method. Expected to outperform M4 on L1–L5 (robustness)
and on Ctrl (OOD generalisation) while maintaining L0 parity.


In [ ]:
M6_ADAPTER = "checkpoints/inv_grpo_final"
M6_OUTPUT  = "results/M6_inv_grpo"

m6_report = suite.evaluate_model(
    model_id="M6_inv_grpo",
    model_name_or_path=BASE_MODEL,
    adapter_path=M6_ADAPTER,
    output_dir=M6_OUTPUT,
)
print(f"\nM6 Ladder AUC     : {m6_report.ladder_auc*100:.2f}%")
print(f"M6 Collapse Point : {m6_report.collapse_point}")


## Cell 07 — Load Cached Evaluation Reports

Reports are saved as `results/<model_id>/eval_report.json`.
Loading from disk means evaluation can be run once and re-analysed freely.


In [ ]:
import json
from pathlib import Path

REPORT_PATHS = {
    "M1_baseline":      "results/M1_baseline/M1_baseline/eval_report.json",
    "M2_vanilla_sft":   "results/M2_vanilla_sft/M2_vanilla_sft/eval_report.json",
    "M4_standard_grpo": "results/M4_standard_grpo/M4_standard_grpo/eval_report.json",
    "M6_inv_grpo":      "results/M6_inv_grpo/M6_inv_grpo/eval_report.json",
}

all_reports = {}
for model_id, path in REPORT_PATHS.items():
    p = Path(path)
    if p.exists():
        with open(p) as f:
            all_reports[model_id] = json.load(f)
        print(f"✅ Loaded {model_id} from {p}")
    else:
        print(f"⚠️  Missing: {p}")

print(f"\nLoaded {len(all_reports)} / {len(REPORT_PATHS)} reports")


## Cell 08 — Master Comparison Table (All Models × All Metrics)

In [ ]:
from src.evaluation import EvaluationReporter
import pandas as pd

reporter = EvaluationReporter(
    reports=all_reports,
    output_dir="results/figures",
)

master_table = reporter.build_master_table()

# Display in notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
display(master_table)

# Also save as CSV
master_table.to_csv("results/master_comparison_table.csv")
print("\n✅ Saved → results/master_comparison_table.csv")


## Cell 09 — Fig 1: 4-Curve Degradation Plot (L0→Ctrl)

In [ ]:
from IPython.display import Image

fig1_path = reporter.plot_degradation_curves()
Image(str(fig1_path), width=900)


## Cell 10 — Fig 2: Radar Chart (Multi-Axis Model Fingerprints)

In [ ]:
fig2_path = reporter.plot_radar_chart()
Image(str(fig2_path), width=600)


## Cell 11 — Fig 3: Grouped Bar Chart (Pass@1 per Level) + Fig 4: Heatmap

In [ ]:
from IPython.display import display as ipy_display

fig3_path = reporter.plot_grouped_bars()
fig4_path = reporter.plot_metric_heatmap()

ipy_display(Image(str(fig3_path), width=1000))
ipy_display(Image(str(fig4_path), width=900))


## Cell 12 — Scientific Conclusions

> **This section is auto-populated.** Run the cell to generate validated conclusions
> directly from the metric numbers in `master_table`.


In [ ]:
# ── Extract key numbers ───────────────────────────────────────────────────────
def pct(s):
    try: return float(str(s).replace('%','').replace('+','').strip())
    except: return 0.0

def gain(model, metric):
    m6 = pct(master_table.loc[model, metric])
    m1 = pct(master_table.loc['M1_baseline', metric])
    return m6 - m1

print("=" * 70)
print("SCIENTIFIC CONCLUSIONS — REDUCTION LADDER FOR CODE")
print("=" * 70)

if 'M6_inv_grpo' in master_table.index and 'M1_baseline' in master_table.index:
    auc_gain = gain('M6_inv_grpo', 'Ladder AUC')
    l5_gain  = pct(master_table.loc['M6_inv_grpo','L5']) - pct(master_table.loc['M1_baseline','L5'])
    l0_delta = pct(master_table.loc['M6_inv_grpo','L0']) - pct(master_table.loc['M1_baseline','L0'])

    print(f"\n1. Ladder AUC Improvement (M6 vs M1): {auc_gain:+.1f} pp")
    print(f"   Interpretation: Inv-GRPO raises overall robustness across all 6 levels.")

    print(f"\n2. L5 (Hardest Level) Gain (M6 vs M1): {l5_gain:+.1f} pp")
    print(f"   Interpretation: Inv-GRPO generalises to the most complex transformations.")

    print(f"\n3. L0 (HumanEval) Delta (M6 vs M1): {l0_delta:+.1f} pp")
    msg = "No regression at L0." if l0_delta >= -1.0 else "Slight L0 regression — needs analysis."
    print(f"   Interpretation: {msg}")

if 'M4_standard_grpo' in master_table.index and 'M6_inv_grpo' in master_table.index:
    auc_vs_m4 = pct(master_table.loc['M6_inv_grpo','Ladder AUC']) - pct(master_table.loc['M4_standard_grpo','Ladder AUC'])
    print(f"\n4. M6 vs M4 (Ablation — Invariance Term Value): {auc_vs_m4:+.1f} pp AUC")
    if auc_vs_m4 > 0:
        print(f"   Interpretation: Invariance regularisation adds {auc_vs_m4:.1f} pp AUC over Standard GRPO.")
    else:
        print(f"   Interpretation: Invariance regularisation did not improve AUC over Standard GRPO.")

print("\n" + "=" * 70)
